In [1]:
!nvidia-smi

!nvidia-smi

!pip install -U peft==0.10.0
!pip install -U accelerate==0.30.1
!pip install -U transformers datasets bitsandbytes
!pip install -q trl

import os
os.makedirs("logs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("✅ Setup complete!")

Tue Nov 11 05:20:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset
import json

# training_data.json(学習用のデータ)はContent内に格納しておく
dataset = load_dataset("json", data_files="training_data.json")

print("✅ Dataset loaded:", dataset)

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 100
    })
})


In [5]:
def preprocess_function(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    text_pairs = [f"User: {inp}\nBot: {out}" for inp, out in zip(inputs, outputs)]

    model_inputs = tokenizer(
        text_pairs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

processed_dataset = dataset["train"].map(preprocess_function, batched=True)
print("✅ Tokenization complete!")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Tokenization complete!


In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA configuration applied.")

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.04703666202518836
✅ LoRA configuration applied.


In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    tokenizer=tokenizer
)

trainer.train()
print("✅ Training finished.")

/tmp/ipython-input-1296116397.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,4.201700
20,0.117600
30,0.090900
40,0.074900
50,0.074500
60,0.059800
70,0.058200
80,0.050900
90,0.048800
100,0.044600


✅ Training finished.


In [8]:
OUTPUT_DIR = "mistral7b_finetuned"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

✅ Model saved to mistral7b_finetuned


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import gc 

# --- メモリの解放処理 ---
print("➡️ Releasing existing model from VRAM...")

# 以前のモデルオブジェクト（model, base_model, trainerなど）を削除
try:
    del model
    del base_model
    del trainer
    del pipe
except NameError:
    pass

# PyTorchにキャッシュされているVRAMをクリア
torch.cuda.empty_cache()
gc.collect()

print("✅ VRAM cleaned up. Free memory should now be available.")


# --- 設定 ---
LOCAL_ADAPTER_DIR = "mistral7b_finetuned"
BASE_MODEL_ID = "mistralai/Mistral-7B-v0.1"

# --- 1. ベースモデルのロード (訓練時と同じ4-bit量子化を使用) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("➡️ Loading Base Model (4-bit)...")

# ベースモデルをロード
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# --- 2. トークナイザのロード ---
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# --- 3. LoRAアダプターの適用 ---
print("➡️ Applying LoRA Adapter...")
model = PeftModel.from_pretrained(
    base_model,
    LOCAL_ADAPTER_DIR
)

# --- 4. パイプラインのセットアップと推論 ---
# トークナイザとLoRA適用済みモデルで推論パイプラインを作成
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map={'': 0}
)

print("✅ Pipeline loaded and LoRA adapter applied successfully.")

# 推論の実行
prompt = "User: What are the latest trends in artificial intelligence?\nBot:"
result = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)

print("\n💬 Generated Response:")
print(result[0]["generated_text"])

➡️ Releasing existing model from VRAM...
✅ VRAM cleaned up. Free memory should now be available.
➡️ Loading Base Model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

➡️ Applying LoRA Adapter...


Device set to use cuda:0


✅ Pipeline loaded and LoRA adapter applied successfully.

💬 Generated Response:
User: What are the latest trends in artificial intelligence?
Bot: There are many exciting trends in AI, such as advances in machine learning, computer vision, and natural language processing.


In [14]:
!pip install gradio

In [ ]:
import gradio as gr
import torch
from transformers import pipeline

# ----------------------------------------------------------------------
# 処理関数: ユーザー入力と履歴を受け取り、モデルに渡して応答を生成する
# ----------------------------------------------------------------------
def chat_with_model(chat_prompt_text):
    """モデルの推論を実行し、Botの応答テキストを返します。"""
    result = pipe(
        chat_prompt_text,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.8,
        return_full_text=False
    )
    bot_response = result[0]['generated_text'].strip()
    if "User:" in bot_response:
        bot_response = bot_response.split("User:")[0].strip()

    return bot_response


# ----------------------------------------------------------------------
# GUIアクション関数
# ----------------------------------------------------------------------

# 1. ユーザーメッセージを履歴に追加する関数
def add_message(message, history):
    return "", history + [[message, None]]

# 2. モデルに応答を生成させ、履歴に追加する関数
def generate_response(history):
    chat_prompt = ""
    for user_msg, bot_resp in history:
        if bot_resp is None:
             chat_prompt += f"User: {user_msg}\nBot:"
        else:
             chat_prompt += f"User: {user_msg}\nBot: {bot_resp}\n"

    bot_response = chat_with_model(chat_prompt)
    history[-1][1] = bot_response

    return history

# ----------------------------------------------------------------------
# Gradio Blocksを使用したカスタムUIの構築
# ----------------------------------------------------------------------

custom_theme = gr.themes.Soft(
    font=["sans-serif"],
).set(
    body_background_fill='white',
    button_primary_background_fill_hover='*primary_300',
)

custom_css = """
.submit-button button {
    margin-top:10px !important;
}
"""

with gr.Blocks(theme=custom_theme, title="Fine-tuned Mistral Chatbot", css=custom_css) as demo:
    gr.Markdown("# 📚 LoRA Fine-tuned Mistral Chatbot")

    # 1. チャット履歴エリア
    chatbot = gr.Chatbot(height=500, label="チャット履歴")

    # 2. 入力コンポーネントを横並びにする
    with gr.Row():
        txt = gr.Textbox(
            show_label=False,
            placeholder="メッセージを入力してEnterキーを押すか、送信ボタンをクリックしてください...",
            lines=3,
            scale=4
        )

        with gr.Column(scale=1, elem_classes="submit-button"):
            submit_btn = gr.Button("送信", variant="primary")

    clear_btn = gr.ClearButton([txt, chatbot])

    # 3. イベントの定義

    txt_msg = txt.submit(
        fn=add_message,
        inputs=[txt, chatbot],
        outputs=[txt, chatbot],
        queue=False
    )
    btn_msg = submit_btn.click(
        fn=add_message,
        inputs=[txt, chatbot],
        outputs=[txt, chatbot],
        queue=False
    )

    txt_msg.then(
        fn=generate_response,
        inputs=[chatbot],
        outputs=[chatbot],
        queue=False
    )
    btn_msg.then(
        fn=generate_response,
        inputs=[chatbot],
        outputs=[chatbot],
        queue=False
    )


# Gradioインターフェースの起動
demo.launch(share=True)

/tmp/ipython-input-3173113267.py:72: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500, label="チャット履歴")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62f26502539b6aa6c6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
